# MLflow Quick Reference — agent-042

Cheat-sheet for using the **remote MLflow server** (tracking + model registry).

## Architecture

```text
┌─ Jupyter container ──────────────────────────────────────────┐
│  (mlflow_db_net)  →  http://mlflow:5000   ← no auth, fastest │
└──────────────────────────────────────────────────────────────┘

┌─ Your Laptop ────────────────────────────────────────────────┐
│  SSH tunnel  →  http://127.0.0.1:5050     ← no auth          │
│  Public URL  →  https://agent.antonlab.ru:8443/mlflow/       │
│                  + MLFLOW_TRACKING_USERNAME / _PASSWORD      │
└──────────────────────────────────────────────────────────────┘
```

Artifacts live in **Yandex Object Storage** (S3-compatible).  
The client needs S3 credentials in the environment to read/write artifacts directly  
(MLflow server is started **without** `--serve-artifacts`).

If not in the Jupyter container, set these env vars to enable S3 access for MLflow client:

```bash
export MLFLOW_S3_ENDPOINT_URL="https://storage.yandexcloud.net"
export AWS_ACCESS_KEY_ID="..."
export AWS_SECRET_ACCESS_KEY="..."
export AWS_DEFAULT_REGION="ru-central1"
```

---
## 1. Connect to the Server

In [ ]:
import os
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

In [ ]:
PROJECT_ROOT = Path("/home/jovyan")
assert (PROJECT_ROOT / "assets").exists()

In [ ]:
TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]
TRACKING_URI

In [ ]:
"""Setup: load credentials and point the client at the remote server."""
from __future__ import annotations

import os
from pathlib import Path

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

# Verify connection
experiments = client.search_experiments()
print(f"Connected to : {TRACKING_URI}")
print(f"Experiments  : {len(experiments)}")
for exp in experiments[:10]:
    print(exp)

---
## 2. MLflow Tracking — Log an Experiment Run

Use `mlflow.start_run()` to open a context and log whatever you want.  
Useful when running quick experiments from your laptop (e.g. evaluating a local retrieval config).

In [ ]:
import random

# Set the experiment (created automatically if it doesn't exist)
EXPERIMENT_NAME = "laptop-experiments/quickref-demo"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="demo-run") as run:
    # ── Tags: free-form metadata ──────────────────────────────────────────────
    mlflow.set_tags(
        {
            "author": "laptop",
            "task": "retrieval",
            "dataset": "beir-scifact",
        }
    )

    # ── Parameters: hyperparameters, config ───────────────────────────────────
    params = {
        "top_k": 10,
        "reranker": "cross-encoder",
        "embedding_model": "BAAI/bge-m3",
        "chunk_size": 512,
    }
    mlflow.log_params(params)

    # ── Metrics: step-by-step or just a final scalar ──────────────────────────
    for step in range(5):
        noise = random.gauss(0, 0.01)
        mlflow.log_metrics(
            {
                "ndcg@10": round(0.62 + step * 0.01 + noise, 4),
                "recall@10": round(0.74 + step * 0.005 + noise, 4),
            },
            step=step,
        )

    # ── Artifacts: any local file ─────────────────────────────────────────────
    # Write a temporary results file and upload it
    results_path = Path("/tmp/retrieval_results.txt")
    results_path.write_text("ndcg@10=0.647\nrecall@10=0.762\n")
    mlflow.log_artifact(str(results_path), artifact_path="results")

    print(f"Run ID  : {run.info.run_id}")
    print(f"UI link : {TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

---
## 3. Query Runs with MlflowClient

Browse existing experiments and runs programmatically.

In [ ]:
### 3a. List all experiments

experiments = client.search_experiments(order_by=["last_update_time DESC"])
print(f"{'ID':<6} {'Name':<45} {'Stage'}")
print("-" * 65)
for exp in experiments:
    print(f"{exp.experiment_id:<6} {exp.name:<45} {exp.lifecycle_stage}")

In [ ]:
### 3b. Search runs — filter, sort, limit

# Full filter syntax: https://mlflow.org/docs/latest/search-runs.html
SEARCH_EXPERIMENT = "adapters/pretrained-lora"  # change to any experiment name

exp = client.get_experiment_by_name(SEARCH_EXPERIMENT)
if exp is None:
    print(f"Experiment '{SEARCH_EXPERIMENT}' not found.")
    runs = []
else:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="attributes.status = 'FINISHED'",
        order_by=["start_time DESC"],
        max_results=10,
    )
    print(f"Found {len(runs)} run(s)\n")
    for r in runs:
        print(
            f"  run_id={r.info.run_id[:8]}…  "
            f"name={r.info.run_name or '—':<20}  "
            f"status={r.info.status}"
        )

In [ ]:
### 3c. Inspect a specific run

RUN_ID = runs[0].info.run_id if runs else None  # replace with a real run ID

if RUN_ID:
    run = client.get_run(RUN_ID)
    print("=== Info ===")
    print(f"  run_id    : {run.info.run_id}")
    print(f"  name      : {run.info.run_name}")
    print(f"  status    : {run.info.status}")
    print(f"  experiment: {run.info.experiment_id}")

    print("\n=== Params ===")
    for k, v in run.data.params.items():
        print(f"  {k}: {v}")

    print("\n=== Metrics (last step) ===")
    for k, v in run.data.metrics.items():
        print(f"  {k}: {v}")

    print("\n=== Tags ===")
    for k, v in run.data.tags.items():
        if not k.startswith("mlflow."):  # skip internal mlflow tags
            print(f"  {k}: {v}")

    print("\n=== Artifacts ===")
    for a in client.list_artifacts(RUN_ID):
        print(f"  {a.path}  (is_dir={a.is_dir})")
else:
    print("No finished runs found — run Section 2 first.")

In [ ]:
### 3d. Download an artifact from a run to your laptop

import mlflow.artifacts

if RUN_ID:
    # Downloads to a local temp folder and returns the path
    local_path = mlflow.artifacts.download_artifacts(
        run_id=RUN_ID,
        artifact_path="results",  # subdirectory inside the run's artifacts
        dst_path="/tmp/mlflow_downloads",
    )
    print(f"Downloaded to: {local_path}")
    for f in Path(local_path).rglob("*"):
        if f.is_file():
            print(f"  {f}  ({f.stat().st_size} bytes)")
            print(f.read_text()[:200])
else:
    print("No run ID available — run Section 2 first.")

---
## 4. Model Registry

The registry holds the LoRA adapters. Lifecycle:

```
run (tracked)
  └─ register_model()  →  "None" status
       └─ set_alias("challenger")   ← under evaluation
            └─ set_alias("champion")  ← production, picked up by vLLM adapter-sync
```

"Adapter names used in this project: `lora-summarize`, `lora-code`"

In [ ]:
### 4a. List all registered models and their versions

registered_models = client.search_registered_models(order_by=["last_updated_timestamp DESC"])

if not registered_models:
    print("No registered models found.")
else:
    for rm in registered_models:
        print(f"\n{'=' * 60}")
        print(f"  Model      : {rm.name}")
        print(f"  Description: {(rm.description or '—')[:80]}")
        if rm.tags:
            print(f"  Tags       : {dict(rm.tags)}")

        versions = client.search_model_versions(f"name='{rm.name}'")
        for v in sorted(versions, key=lambda x: int(x.version)):
            aliases = client.get_model_version_by_alias  # just for reference
            alias_list = v.aliases if hasattr(v, "aliases") else []
            print(
                f"    v{v.version:<3}  run={v.run_id[:8]}…  "
                f"status={v.status:<10}  aliases={alias_list}"
            )

In [ ]:
### 4b. Fetch a model version by alias

MODEL_NAME = "lora-summarize"  # change as needed

for alias in ("champion", "challenger"):
    try:
        mv = client.get_model_version_by_alias(MODEL_NAME, alias)
        print(f"[{alias}]  version={mv.version}  run={mv.run_id[:8]}…  source={mv.source}")
    except Exception:
        print(f"[{alias}]  — not set")

In [ ]:
### 4c. Register a model from an existing run
#
# Point at the run artifact folder that contains the adapter weights.
# After registering, the version starts with no aliases.

# RUN_ID = "paste-a-real-run-id-here"
# model_uri = f"runs:/{RUN_ID}/adapter"   # artifact_path used when logging

# mv = mlflow.register_model(
#     model_uri=model_uri,
#     name=MODEL_NAME,
#     tags={"task": "summarization", "base_model": "Qwen/Qwen3-0.6B"},
# )
# print(f"Registered: {mv.name}  version={mv.version}")

print("Uncomment and fill in RUN_ID to register a new version.")

In [ ]:
### 4d. Promote (set alias) — move a version to challenger or champion
#
# This is the step that triggers the vLLM adapter-sync container on the server
# to pick up the new adapter when it next runs.

# VERSION = "3"
# client.set_registered_model_alias(MODEL_NAME, "challenger", VERSION)
# print(f"Set '{MODEL_NAME}' v{VERSION} → challenger")

# # After eval looks good — promote to production:
# client.set_registered_model_alias(MODEL_NAME, "champion", VERSION)
# print(f"Set '{MODEL_NAME}' v{VERSION} → champion")

# # Remove an alias if needed:
# client.delete_registered_model_alias(MODEL_NAME, "challenger")

print("Uncomment the lines above and set VERSION to promote an adapter.")

In [ ]:
### 4e. Download the champion adapter to your laptop
#
# Requires S3 credentials in env (MLFLOW_S3_ENDPOINT_URL, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY).

# champion_mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
# local_adapter_dir = Path(f"/tmp/adapters/{MODEL_NAME}_v{champion_mv.version}")
#
# local_adapter_dir = mlflow.artifacts.download_artifacts(
#     artifact_uri=champion_mv.source,
#     dst_path=str(local_adapter_dir),
# )
# print(f"Champion adapter downloaded to: {local_adapter_dir}")
# for f in Path(local_adapter_dir).rglob("*"):
#     if f.is_file():
#         print(f"  {f.relative_to(local_adapter_dir)}  ({f.stat().st_size:,} bytes)")

print("Uncomment to download the champion adapter locally.")

## Notes

`src/services/adapter_sync/model_registry.py` is the canonical Python API for adapter registry operations.

`MLFLOW_BACKEND_URI` is reserved for the MLflow server backend store configuration.
Client-side code and notebooks should use `MLFLOW_TRACKING_URI` when connecting to MLflow.
For interactive registry operations, prefer `experiments/training/lora_ops.ipynb`, which reflects the current adapter workflow.